In [1]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q timm openpyxl
import sys, torch, timm
print(sys.version)
print("torch", torch.__version__, "| timm", timm.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU")

Mounted at /content/drive
3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
torch 2.11.0+cu130 | timm 1.0.29
GPU: Tesla T4


In [2]:
!git clone https://github.com/VinUni-AI20k/K4-Track4-Day2-Deeplearning-Advance.git
!ls K4-Track4-Day2-Deeplearning-Advance
!ls K4-Track4-Day2-Deeplearning-Advance/starter

Cloning into 'K4-Track4-Day2-Deeplearning-Advance'...
remote: Enumerating objects: 34, done.
remote: Counting objects: 100% (8/8), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 34 (delta 1), reused 0 (delta 0), pack-reused 26 (from 1)
Receiving objects: 100% (34/34), 68.45 KiB | 2.85 MiB/s, done.
Resolving deltas: 100% (7/7), done.
eval.py  GUIDE.md  README.md  RUBRIC.md  starter  tests
benchmark.py  inference.py    losses.py  train.py
dataset.py    lab_day2.ipynb  model.py


In [3]:
!mkdir -p /content/data
!wget -q -O /content/data/images.zip "https://zenodo.org/records/7939060/files/images.zip?download=1"
!md5sum /content/data/images.zip
# phải ra: b7b30f96d466fba86016aa5a26606e0f

b7b30f96d466fba86016aa5a26606e0f  /content/data/images.zip


In [4]:
!cd /content/data && unzip -q images.zip
base = "https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels"
for f in ["labels.csv","train_subset0.csv","val_subset0.csv","test_subset0.csv"]:
    !wget -q -P /content/data {base}/{f}
!ls /content/data
!find /content/data -name "*.jpg" | head -3

Kết quả truyền trực tuyến bị cắt bớt đến 5000 dòng cuối.
20170210-145509-0.jpg  20171102-115316-1.jpg  20180109-072347-2.jpg
20170210-145656-0.jpg  20171102-115324-2.jpg  20180109-072350-1.jpg
20170210-145705-0.jpg  20171102-115333-3.jpg  20180109-072359-2.jpg
20170210-145719-0.jpg  20171102-115337-1.jpg  20180109-072402-1.jpg
20170210-145749-0.jpg  20171102-115337-2.jpg  20180109-072410-2.jpg
20170210-145824-0.jpg  20171102-115350-1.jpg  20180109-072414-1.jpg
20170210-150032-0.jpg  20171102-115408-3.jpg  20180109-072422-2.jpg
20170210-150041-0.jpg  20171102-115409-1.jpg  20180109-072426-1.jpg
20170210-150054-0.jpg  20171102-115412-2.jpg  20180109-072434-2.jpg
20170210-150110-0.jpg  20171102-115438-1.jpg  20180109-072438-1.jpg
20170210-150120-0.jpg  20171102-115438-2.jpg  20180109-072442-2.jpg
20170210-150136-0.jpg  20171102-115445-3.jpg  20180109-072448-1.jpg
20170210-150223-0.jpg  20171102-115453-2.jpg  20180109-072451-2.jpg
20170210-150245-0.jpg  20171102-115510-3.jpg  20180109-0725

In [6]:
import pandas as pd
tr = pd.read_csv("/content/data/train_subset0.csv")
va = pd.read_csv("/content/data/val_subset0.csv")
te = pd.read_csv("/content/data/test_subset0.csv")
print(len(tr), len(va), len(te))
print(tr.Label.value_counts())

10501 3501 3507
Label
8    5463
0     675
6     644
4     637
1     637
2     618
3     613
7     609
5     605
Name: count, dtype: int64


In [8]:
!find /content/data -name "20171109-175921-2.jpg"
!ls /content/data | head
!ls /content/data/images | head -3

/content/data/20171109-175921-2.jpg
20160928-140314-0.jpg
20160928-140337-0.jpg
20160928-140731-0.jpg
20160928-140747-0.jpg
20160928-141107-0.jpg
20160928-141135-0.jpg
20160928-141355-0.jpg
20160928-141421-0.jpg
20160928-141437-0.jpg
20160928-142056-0.jpg
ls: cannot access '/content/data/images': No such file or directory


In [9]:
!ls -lh /content/data/images.zip
!md5sum /content/data/images.zip
!cd /content/data && unzip -q -o images.zip
!find /content/data -name "*.jpg" | wc -l   # phải ra 17509

-rw-r--r-- 1 root root 469M Oct  4 17:38 /content/data/images.zip
b7b30f96d466fba86016aa5a26606e0f  /content/data/images.zip
17509


In [11]:
import os, glob
p = glob.glob("/content/data/**/20171109-175921-2.jpg", recursive=True)
print(p)
IMAGES_DIR = os.path.dirname(p[0])
print("IMAGES_DIR =", IMAGES_DIR)
print(os.path.exists(f"{IMAGES_DIR}/{tr.Filename[0]}"))   # phải ra True

['/content/data/20171109-175921-2.jpg']
IMAGES_DIR = /content/data
True


In [12]:
R = "K4-Track4-Day2-Deeplearning-Advance"
!ls -R {R} | head -50
!cat {R}/README.md

K4-Track4-Day2-Deeplearning-Advance:
eval.py
GUIDE.md
README.md
RUBRIC.md
starter
tests

K4-Track4-Day2-Deeplearning-Advance/starter:
benchmark.py
dataset.py
inference.py
lab_day2.ipynb
losses.py
model.py
train.py

K4-Track4-Day2-Deeplearning-Advance/tests:
test_eval.py
test_starter.py
# Lab Day 2 — Backbone, công thức huấn luyện và suy luận trên DeepWeeds

> Track 4 · Ngày 2 · *Tích chập, chuỗi, attention · backbone · huấn luyện · suy luận*
> Bài lab này mở rộng **Lab #2** trong slide Day 2. Slide chỉ yêu cầu 1 backbone, 3 cách khởi tạo, có/không CutMix và TTA. Ở đây bạn làm đầy đủ: **≥ 5 backbone**, **nhiều công thức huấn luyện**, **nhiều cách suy luận**, rồi chọn cấu hình tốt nhất và báo cáo.

Repo gồm **hướng dẫn, tiêu chí chấm, bộ khung code (pseudo-code) và công cụ đánh giá**. Bộ khung `starter/` chỉ có chữ ký hàm, docstring và các bước `TODO`: **bạn tự viết phần ruột** (model, loss, augmentation, vòng huấn luyện, TTA, đo độ trễ). Riêng `eval.py` đã hoàn chỉnh, bạn không sửa. Làm

In [13]:
import os
IMAGES_DIR = "/content/data"
OUT = "/content/drive/MyDrive/deepweeds"
os.makedirs(OUT, exist_ok=True)

In [14]:
import pandas as pd, os
D = "/content/data"
tr = pd.read_csv(f"{D}/train_subset0.csv"); va = pd.read_csv(f"{D}/val_subset0.csv"); te = pd.read_csv(f"{D}/test_subset0.csv")
s = lambda d: set(d.Filename)
print(len(tr), len(va), len(te), len(tr)+len(va)+len(te))
print("giao:", len(s(tr)&s(va)), len(s(tr)&s(te)), len(s(va)&s(te)))   # phải 0 0 0
print("thiếu file:", sum(not os.path.exists(f"{D}/{f}") for f in pd.concat([tr,va,te]).Filename))
for n,d in [("train",tr),("val",va),("test",te)]: print(n, d.Label.value_counts().sort_index().tolist())

10501 3501 3507 17509
giao: 0 0 0
thiếu file: 0
train [675, 637, 618, 613, 637, 605, 644, 609, 5463]
val [225, 213, 206, 204, 212, 202, 215, 203, 1821]
test [226, 213, 207, 205, 213, 202, 215, 204, 1822]


In [15]:
R = "K4-Track4-Day2-Deeplearning-Advance"
!cat {R}/GUIDE.md
!cat {R}/RUBRIC.md

# GUIDE — Quy trình thực hiện Lab Day 2

Hướng dẫn này đi theo thứ tự bạn nên làm. Hãy đọc hết một lượt trước khi bắt đầu. Các số liệu và trích dẫn từ slide Day 2 ghi kèm số trang; mọi mốc thời gian GPU trong tài liệu này là **ước lượng chưa được đo**, hãy tự đo epoch đầu tiên rồi tính lại.

## Mục lục

0. [Nguyên tắc thực nghiệm (đọc kỹ)](#0-nguyên-tắc-thực-nghiệm)
1. [Bước 0: Chuẩn bị, EDA, dựng pipeline](#1-bước-0-chuẩn-bị-eda-dựng-pipeline)
2. [Bước 1: So sánh backbone](#2-bước-1-so-sánh-backbone--5)
3. [Bước 2: Công thức huấn luyện](#3-bước-2-công-thức-huấn-luyện)
4. [Bước 3: Phương pháp suy luận](#4-bước-3-phương-pháp-suy-luận)
5. [Bước 4: Cấu hình tốt nhất và chạy test](#5-bước-4-cấu-hình-tốt-nhất-và-chạy-test)
6. [Bước 5: Sản phẩm (xlsx, biểu đồ, báo cáo)](#6-bước-5-sản-phẩm)
7. [Ngân sách tính toán và mẹo Colab/Kaggle](#7-ngân-sách-tính-toán-và-mẹo-colabkaggle)
8. [Bẫy thường gặp](#8-bẫy-thường-gặp)
9. [Câu hỏi gợi ý cho phần phân tích](#9-câu-hỏi-gợi-ý-cho-phần-phân-tích)

--

In [16]:
R = "K4-Track4-Day2-Deeplearning-Advance"
!cat {R}/starter/dataset.py
!cat {R}/starter/model.py
!cat {R}/starter/train.py

"""dataset.py - đọc DeepWeeds, kiểm tra chia dữ liệu, transform, DataLoader.

PSEUDO-CODE: bạn tự hoàn thiện mọi hàm có `raise NotImplementedError`.
Quy tắc chia dữ liệu bắt buộc (S1-S6) nằm ở README.md, mục 2.1. Đọc trước khi viết.

Giao diện bạn phải giữ (để notebook, train.py và eval.py ghép được với nhau):
    load_split(labels_dir, fold=0)            -> (train_df, val_df, test_df)
    check_split(train_df, val_df, test_df, images_dir) -> dict  (số liệu để ghi báo cáo)
    build_transforms(train, img_size, aug)    -> torchvision transform
    DeepWeedsDataset[i]                       -> (image_tensor, label:int, filename:str)
    make_loader(df, images_dir, transform, batch_size, train, sampler, num_workers)
"""
from __future__ import annotations

from pathlib import Path

import pandas as pd

NUM_CLASSES = 9
# Thứ tự lớp theo cột `Label` của labels.csv (0 = Chinee Apple ... 7 = Snake Weed, 8 = Negatives).
CLASS_NAMES = [
    "Chinee Apple", "Lantana", "Parkinsonia", "Parthenium", 

In [17]:
OUT = "/content/drive/MyDrive/deepweeds"
CODE = f"{OUT}/code"
import os
os.makedirs(CODE, exist_ok=True)
print(os.listdir(OUT))

['code']


In [19]:
%%writefile /content/drive/MyDrive/deepweeds/code/dataset.py
%%writefile {CODE}/dataset.py
from __future__ import annotations
from pathlib import Path
import numpy as np, pandas as pd, torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms as T

NUM_CLASSES = 9
CLASS_NAMES = ["Chinee Apple","Lantana","Parkinsonia","Parthenium","Prickly Acacia",
               "Rubber Vine","Siam Weed","Snake Weed","Negatives"]
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


def load_split(labels_dir, fold=0):
    d = Path(labels_dir)
    return tuple(pd.read_csv(d / f"{s}_subset{fold}.csv") for s in ("train", "val", "test"))


def check_split(train_df, val_df, test_df, images_dir):
    sets = {"train": train_df, "val": val_df, "test": test_df}
    n = {k: len(v) for k, v in sets.items()}
    total = sum(n.values())
    per_class = {k: v.Label.value_counts().sort_index().to_dict() for k, v in sets.items()}
    S = {k: set(v.Filename) for k, v in sets.items()}
    overlap = {"train&val": len(S["train"] & S["val"]),
               "train&test": len(S["train"] & S["test"]),
               "val&test": len(S["val"] & S["test"])}
    union = len(S["train"] | S["val"] | S["test"])
    missing = sum(not (Path(images_dir) / f).exists()
                  for f in pd.concat(sets.values()).Filename)
    ratio = {k: round(v / total, 4) for k, v in n.items()}
    print("n:", n, "| tổng:", total, "| tỉ lệ:", ratio)
    for k, v in per_class.items(): print(k, v)
    print("giao:", overlap, "| hợp:", union, "| thiếu file:", missing)
    assert all(v == 0 for v in overlap.values()), "Giao khác rỗng!"
    assert union == 17509 and total == 17509, "Hợp != 17509"
    assert missing == 0, "Có file không tồn tại"
    assert all(abs(ratio[k] - r) < 0.01 for k, r in zip(("train","val","test"), (.6,.2,.2))), "Lệch 60/20/20"
    return {"n": n, "ratio": ratio, "per_class": per_class, "overlap": overlap, "union": union}


def build_transforms(train, img_size=224, aug="basic", mean=IMAGENET_MEAN, std=IMAGENET_STD):
    norm = [T.ToTensor(), T.Normalize(mean, std)]
    if not train:
        # ảnh gốc 256: <256 -> CenterCrop, ==256 giữ nguyên, >256 -> Resize
        if img_size < 256: pre = [T.CenterCrop(img_size)]
        elif img_size == 256: pre = []
        else: pre = [T.Resize(img_size, interpolation=T.InterpolationMode.BICUBIC)]
        return T.Compose(pre + norm)
    base = [T.RandomResizedCrop(img_size, scale=(0.35, 1.0)), T.RandomHorizontalFlip()]
    if aug == "basic": extra = []
    elif aug == "color": extra = [T.ColorJitter(0.3, 0.3, 0.3, 0.05)]
    elif aug == "trivial": extra = [T.TrivialAugmentWide()]
    elif aug == "randaug": extra = [T.RandAugment(num_ops=2, magnitude=9)]
    else: raise ValueError(f"aug không hợp lệ: {aug}")
    return T.Compose(base + extra + norm)


class DeepWeedsDataset(Dataset):
    def __init__(self, df, images_dir, transform=None, cache=False):
        self.df = df.reset_index(drop=True)
        self.dir = Path(images_dir)
        self.transform = transform
        self.files = self.df.Filename.tolist()
        self.labels = self.df.Label.astype(int).tolist()
        self.cache = None
        if cache:  # nạp trước uint8 vào RAM (~65GB/1000 ảnh? không: ~196KB/ảnh)
            self.cache = [np.asarray(Image.open(self.dir / f).convert("RGB")) for f in self.files]

    def __len__(self): return len(self.files)

    def __getitem__(self, i):
        if self.cache is not None: img = Image.fromarray(self.cache[i])
        else: img = Image.open(self.dir / self.files[i]).convert("RGB")
        if self.transform: img = self.transform(img)
        return img, self.labels[i], self.files[i]


def _worker_init(worker_id):
    s = torch.initial_seed() % 2**32
    np.random.seed(s); import random; random.seed(s)


def make_loader(df, images_dir, transform, batch_size, train, sampler=None,
                num_workers=2, seed=0, cache=False):
    ds = DeepWeedsDataset(df, images_dir, transform, cache=cache)
    g = torch.Generator(); g.manual_seed(seed)
    kw = dict(batch_size=batch_size, num_workers=num_workers, pin_memory=True,
              worker_init_fn=_worker_init, generator=g,
              persistent_workers=num_workers > 0)
    if train:
        if sampler == "balanced":
            cnt = np.bincount(ds.labels, minlength=NUM_CLASSES)
            w = torch.tensor([1.0 / cnt[l] for l in ds.labels], dtype=torch.double)
            smp = WeightedRandomSampler(w, num_samples=len(ds), replacement=True, generator=g)
            return DataLoader(ds, sampler=smp, drop_last=True, **kw)
        return DataLoader(ds, shuffle=True, drop_last=True, **kw)
    return DataLoader(ds, shuffle=False, drop_last=False, **kw)

Overwriting /content/drive/MyDrive/deepweeds/code/dataset.py


In [20]:
%%writefile /content/drive/MyDrive/deepweeds/code/model.py
%%writefile {CODE}/model.py
from __future__ import annotations
import timm, torch, torch.nn as nn

SUGGESTED_BACKBONES = {
    "resnet50": "resnet50", "resnext50": "resnext50_32x4d", "convnext_tiny": "convnext_tiny",
    "deit_small": "deit_small_patch16_224", "swin_tiny": "swin_tiny_patch4_window7_224",
    "efficientnet_b0": "efficientnet_b0", "mobilenetv3": "mobilenetv3_large_100",
}


def _head_params(model):
    return list(model.get_classifier().parameters())


def build_model(name, pretrained=True, num_classes=9, drop_rate=0.0, init="finetune"):
    name = SUGGESTED_BACKBONES.get(name, name)
    use_pre = init != "scratch" and pretrained
    model = timm.create_model(name, pretrained=use_pre, num_classes=num_classes, drop_rate=drop_rate)
    cfg = getattr(model, "pretrained_cfg", {}) or {}
    model.weights_tag = (cfg.get("hf_hub_id") or cfg.get("url") or "none") if use_pre else "scratch"
    model.arch_name = name
    model.init_mode = init
    if init == "frozen":
        freeze_backbone(model)
    return model


def freeze_backbone(model):
    head = {id(p) for p in _head_params(model)}
    for p in model.parameters():
        p.requires_grad = id(p) in head
    model.init_mode = "frozen"


def set_train_mode(model):
    """Gọi thay cho model.train() trong train loop: nếu frozen thì backbone ở eval (BN không đổi)."""
    model.train()
    if getattr(model, "init_mode", "") == "frozen":
        head = {id(p) for p in _head_params(model)}
        for m in model.modules():
            if not any(id(p) in head for p in m.parameters(recurse=False)) and \
               not list(m.children()):
                if isinstance(m, nn.modules.batchnorm._BatchNorm):
                    m.eval()
        # BN nằm trong head (nếu có) vẫn train; backbone BN đã eval ở trên


def param_groups(model, lr_backbone, lr_head, weight_decay):
    head = {id(p) for p in _head_params(model)}
    bb_decay, bb_nodecay, hd = [], [], []
    for p in model.parameters():
        if not p.requires_grad: continue
        if id(p) in head: hd.append(p)
        elif p.ndim <= 1: bb_nodecay.append(p)
        else: bb_decay.append(p)
    groups = [{"params": bb_decay, "lr": lr_backbone, "weight_decay": weight_decay},
              {"params": bb_nodecay, "lr": lr_backbone, "weight_decay": 0.0},
              {"params": hd, "lr": lr_head, "weight_decay": weight_decay}]
    return [g for g in groups if g["params"]]


def count_params(model):
    return sum(p.numel() for p in model.parameters()) / 1e6


def count_gmacs(model, img_size=224):
    # pip install fvcore ; fvcore đếm 1 MAC = 1 "flop" nên chính là GMAC
    from fvcore.nn import FlopCountAnalysis
    dev = next(model.parameters()).device
    was_training = model.training
    model.eval()
    with torch.inference_mode():
        fa = FlopCountAnalysis(model, torch.randn(1, 3, img_size, img_size, device=dev))
        fa.unsupported_ops_warnings(False); fa.uncalled_modules_warnings(False)
        g = fa.total() / 1e9
    model.train(was_training)
    return g


Writing /content/drive/MyDrive/deepweeds/code/model.py


In [22]:
import re
for name in ["dataset.py", "model.py"]:
    p = f"/content/drive/MyDrive/deepweeds/code/{name}"
    try:
        lines = open(p, encoding="utf-8").read().split("\n")
    except FileNotFoundError:
        print(name, "chưa có"); continue
    if lines[0].startswith("%%writefile"):
        lines = lines[1:]
        open(p, "w", encoding="utf-8").write("\n".join(lines))
        print(name, "đã xóa dòng %%writefile")
    else:
        print(name, "dòng đầu ổn:", lines[0][:50])

dataset.py đã xóa dòng %%writefile
model.py đã xóa dòng %%writefile


In [23]:
!head -3 /content/drive/MyDrive/deepweeds/code/dataset.py
!head -3 /content/drive/MyDrive/deepweeds/code/model.py

from __future__ import annotations
from pathlib import Path
import numpy as np, pandas as pd, torch
from __future__ import annotations
import timm, torch, torch.nn as nn



In [24]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q timm fvcore
import sys
sys.path.insert(0, "/content/drive/MyDrive/deepweeds/code")
import dataset as D, model as M
tr, va, te = D.load_split("/content/data", 0)
info = D.check_split(tr, va, te, "/content/data")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
n: {'train': 10501, 'val': 3501, 'test': 3507} | tổng: 17509 | tỉ lệ: {'train': 0.5997, 'val': 0.2, 'test': 0.2003}
train {0: 675, 1: 637, 2: 618, 3: 613, 4: 637, 5: 605, 6: 644, 7: 609, 8: 5463}
val {0: 225, 1: 213, 2: 206, 3: 204, 4: 212, 5: 202, 6: 215, 7: 203, 8: 1821}
test {0: 226, 1: 213, 2: 207, 3: 205, 4: 213, 5: 202, 6: 215, 7: 204, 8: 1822}
giao: {'train&val': 0, 'train&test': 0, 'val&test': 0} | hợp: 17509 | thiếu file: 0


In [25]:
import torch, math
m = M.build_model("resnet50", True, 9)
print("tag trọng số:", m.weights_tag)
print("params (M):", round(M.count_params(m), 2))
print("GMAC:", round(M.count_gmacs(m), 2))

model.safetensors: reconstructing file:   0%|          |  0.00B /  102MB            

model.safetensors: downloading bytes:           |  0.00B            

tag trọng số: timm/resnet50.a1_in1k
params (M): 23.53
GMAC: 4.11


In [26]:
ld = D.make_loader(tr, "/content/data", D.build_transforms(True), 32, True, num_workers=2)
x, y, f = next(iter(ld))
m = m.cuda().eval()
with torch.inference_mode():
    loss = torch.nn.functional.cross_entropy(m(x.cuda()), y.cuda()).item()
print("loss ban đầu:", loss, "| ln9 =", math.log(9))
print(x.shape, y[:10].tolist(), f[:3])

loss ban đầu: 2.1981654167175293 | ln9 = 2.1972245773362196
torch.Size([32, 3, 224, 224]) [8, 8, 8, 0, 8, 3, 7, 8, 1, 8] ('20171219-104654-2.jpg', '20180109-105931-1.jpg', '20180119-110513-2.jpg')


In [28]:
%%writefile /content/drive/MyDrive/deepweeds/code/losses.py
from __future__ import annotations
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


class LabelSmoothingCE(nn.Module):
    """q'(k) = (1-eps)*1[k==y] + eps/K. Tự cài đặt; eps=0 cho đúng CE."""
    def __init__(self, smoothing: float = 0.1):
        super().__init__()
        self.eps = smoothing

    def forward(self, logits, target):
        logp = F.log_softmax(logits.float(), dim=-1)
        nll = -logp.gather(1, target.unsqueeze(1)).squeeze(1)
        smooth = -logp.mean(dim=-1)
        return ((1 - self.eps) * nll + self.eps * smooth).mean()


class FocalLoss(nn.Module):
    """FL = -alpha_t * (1-p_t)^gamma * log(p_t). gamma=0 (alpha=None) cho đúng CE."""
    def __init__(self, gamma: float = 2.0, alpha=None):
        super().__init__()
        self.gamma = gamma
        if alpha is not None:
            self.register_buffer("alpha", torch.as_tensor(alpha, dtype=torch.float32))
        else:
            self.alpha = None

    def forward(self, logits, target):
        logp = F.log_softmax(logits.float(), dim=-1)
        logpt = logp.gather(1, target.unsqueeze(1)).squeeze(1)
        pt = logpt.exp()
        loss = -((1 - pt) ** self.gamma) * logpt
        if self.alpha is not None:
            loss = loss * self.alpha.to(loss.device)[target]
        return loss.mean()


def class_weights(counts, beta: float = 0.0):
    """Chỉ truyền số ảnh của TRAIN. beta=0: 1/n chuẩn hoá trung bình 1.
    beta>0: class-balanced (1-beta)/(1-beta^n), chuẩn hoá tổng = số lớp."""
    n = np.asarray(counts, dtype=np.float64)
    if beta == 0:
        w = 1.0 / n
        w = w / w.mean()
    else:
        w = (1.0 - beta) / (1.0 - np.power(beta, n))
        w = w / w.sum() * len(n)
    return torch.tensor(w, dtype=torch.float32)


def build_criterion(kind: str = "ce", **kw):
    """kind: ce | ls | focal | ce_weighted.
    kw: smoothing, gamma, alpha, weight (tensor)."""
    if kind == "ce":
        return nn.CrossEntropyLoss()
    if kind == "ls":
        return LabelSmoothingCE(kw.get("smoothing", 0.1))
    if kind == "focal":
        return FocalLoss(kw.get("gamma", 2.0), kw.get("alpha"))
    if kind == "ce_weighted":
        return nn.CrossEntropyLoss(weight=kw["weight"])
    raise ValueError(f"loss không hợp lệ: {kind}")


def mix_batch(x, y, alpha: float = 1.0, mode: str = "cutmix"):
    lam = float(np.random.beta(alpha, alpha))
    perm = torch.randperm(x.size(0), device=x.device)
    if mode == "mixup":
        x_mix = lam * x + (1 - lam) * x[perm]
    elif mode == "cutmix":
        H, W = x.shape[2], x.shape[3]
        cut = np.sqrt(1.0 - lam)
        ch, cw = int(H * cut), int(W * cut)
        cy, cx = np.random.randint(H), np.random.randint(W)
        y1, y2 = np.clip(cy - ch // 2, 0, H), np.clip(cy + ch // 2, 0, H)
        x1, x2 = np.clip(cx - cw // 2, 0, W), np.clip(cx + cw // 2, 0, W)
        x_mix = x.clone()
        x_mix[:, :, y1:y2, x1:x2] = x[perm][:, :, y1:y2, x1:x2]
        lam = 1.0 - float((y2 - y1) * (x2 - x1)) / (H * W)  # diện tích thực sau khi cắt biên
    else:
        raise ValueError(f"mode không hợp lệ: {mode}")
    return x_mix, (y, y[perm], lam)


def mixed_loss(criterion, logits, targets):
    y_a, y_b, lam = targets
    return lam * criterion(logits, y_a) + (1 - lam) * criterion(logits, y_b)

Writing /content/drive/MyDrive/deepweeds/code/losses.py


In [29]:
import sys, importlib
sys.path.insert(0, "/content/drive/MyDrive/deepweeds/code")
import losses as L
importlib.reload(L)
import torch, torch.nn.functional as F

torch.manual_seed(0)
lg = torch.randn(16, 9); y = torch.randint(0, 9, (16,))
ce = F.cross_entropy(lg, y)
print("CE            :", ce.item())
print("focal g=0     :", L.FocalLoss(0.0)(lg, y).item(), "| sai số", abs(L.FocalLoss(0.0)(lg, y) - ce).item())
print("LS eps=0      :", L.LabelSmoothingCE(0.0)(lg, y).item())
print("LS khớp torch :", L.LabelSmoothingCE(0.1)(lg, y).item(), F.cross_entropy(lg, y, label_smoothing=0.1).item())
print("focal g=2     :", L.FocalLoss(2.0)(lg, y).item(), "(phải nhỏ hơn CE)")

w = L.class_weights([675,637,618,613,637,605,644,609,5463], beta=0.0)
print("trọng số lớp  :", w.round(decimals=3).tolist(), "| trung bình", w.mean().item())

x = torch.randn(8, 3, 224, 224)
xm, (ya, yb, lam) = L.mix_batch(x, torch.arange(8), 1.0, "cutmix")
print("cutmix lam    :", lam, "| shape", xm.shape)
xm, (ya, yb, lam) = L.mix_batch(x, torch.arange(8), 1.0, "mixup")
print("mixup lam     :", lam)

CE            : 2.7303948402404785
focal g=0     : 2.7303948402404785 | sai số 0.0
LS eps=0      : 2.7303948402404785
LS khớp torch : 2.7218027114868164 2.7218029499053955
focal g=2     : 2.372457981109619 (phải nhỏ hơn CE)
trọng số lớp  : [1.0329999923706055, 1.0950000286102295, 1.128999948501587, 1.1380000114440918, 1.0950000286102295, 1.152999997138977, 1.0829999446868896, 1.1449999809265137, 0.12800000607967377] | trung bình 1.0
cutmix lam    : 0.37705277423469385 | shape torch.Size([8, 3, 224, 224])
mixup lam     : 0.7218702822997175


In [30]:
R = "K4-Track4-Day2-Deeplearning-Advance"
!cp {R}/eval.py /content/drive/MyDrive/deepweeds/code/
!grep -n -A30 "def compute_metrics" {R}/eval.py
!grep -n -A30 "def save_predictions" {R}/eval.py

259:def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray, probs: np.ndarray,
260-                    k: int = NUM_CLASSES, bins: int = ECE_BINS) -> dict:
261-    cm = confusion_matrix(y_true, y_pred, k)
262-    pc = per_class(cm)
263-    has_support = pc["support"] > 0
264-    nll = float(-np.log(np.clip(probs[np.arange(len(y_true)), y_true], 1e-12, None)).mean())
265-    return {
266-        "n": int(len(y_true)),
267-        "top1": float((y_true == y_pred).mean()),
268-        "macro_f1": float(pc["f1"].mean()),
269-        "balanced_acc": float(pc["recall"][has_support].mean()),
270-        "ece": ece_score(probs, y_true, bins),
271-        "nll": nll,
272-        "precision": pc["precision"], "recall": pc["recall"], "f1": pc["f1"],
273-        "support": pc["support"], "confusion": cm,
274-    }
275-
276-
277-SCALARS = ("top1", "macro_f1", "balanced_acc", "ece", "nll")
278-VECTORS = ("precision", "recall", "f1")
279-
280-
281-def mean_std(values) -> tuple[float, float]:
282-

In [32]:
%%writefile /content/drive/MyDrive/deepweeds/code/train.py
from __future__ import annotations
import argparse, copy, dataclasses, json, math, random, sys, time
from dataclasses import dataclass
from pathlib import Path
import numpy as np, pandas as pd, torch, torch.nn as nn

sys.path.insert(0, str(Path(__file__).parent) if "__file__" in globals() else ".")
import dataset as D, model as M, losses as L
from eval import compute_metrics, save_predictions


@dataclass
class Config:
    exp_id: str = "T00"
    seed: int = 0
    fold: int = 0
    backbone: str = "resnet50"
    init: str = "finetune"
    drop_rate: float = 0.0
    img_size: int = 224
    aug: str = "basic"
    sampler: str | None = None
    mix: str | None = None
    mix_alpha: float = 1.0
    loss: str = "ce"
    label_smoothing: float = 0.0
    focal_gamma: float = 2.0
    class_weight_beta: float | None = None
    epochs: int = 10
    batch_size: int = 64
    lr_backbone: float = 1e-4
    lr_head: float = 1e-3
    weight_decay: float = 0.05
    warmup_epochs: float = 1.0
    ema_decay: float | None = None
    amp: bool = True
    num_workers: int = 2
    cache: bool = True
    images_dir: str = "/content/data"
    labels_dir: str = "/content/data"
    out_dir: str = "/content/drive/MyDrive/deepweeds/runs"
    pred_dir: str = "/content/drive/MyDrive/deepweeds/predictions"
    curves_dir: str = "/content/drive/MyDrive/deepweeds/curves"
    save_test_predictions: bool = False


def run_dir(cfg): return Path(cfg.out_dir) / cfg.exp_id / f"seed{cfg.seed}"
def pred_path(cfg, split): return Path(cfg.pred_dir) / f"{cfg.exp_id}_seed{cfg.seed}_{split}.csv"


def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)


def build_optimizer(model, cfg):
    return torch.optim.AdamW(M.param_groups(model, cfg.lr_backbone, cfg.lr_head, cfg.weight_decay))


def build_scheduler(optimizer, cfg, steps_per_epoch):
    """Cập nhật theo bước: warmup tuyến tính rồi cosine về ~0."""
    total = cfg.epochs * steps_per_epoch
    warm = max(1, int(cfg.warmup_epochs * steps_per_epoch))
    def f(s):
        if s < warm: return (s + 1) / warm
        return 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm)))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, f)


class EMA:
    def __init__(self, model, decay):
        self.decay = decay
        self.module = copy.deepcopy(model).eval()
        for p in self.module.parameters(): p.requires_grad = False

    @torch.no_grad()
    def update(self, model):
        ms, es = model.state_dict(), self.module.state_dict()
        for k, v in es.items():
            if v.dtype.is_floating_point: v.mul_(self.decay).add_(ms[k].detach(), alpha=1 - self.decay)
            else: v.copy_(ms[k])  # num_batches_tracked, BN buffer số nguyên


def build_crit(cfg, train_df, device):
    kw = {}
    if cfg.loss == "ls": kw["smoothing"] = cfg.label_smoothing or 0.1
    if cfg.loss == "focal": kw["gamma"] = cfg.focal_gamma
    if cfg.loss == "ce_weighted":
        cnt = np.bincount(train_df.Label.values, minlength=9)
        kw["weight"] = L.class_weights(cnt, cfg.class_weight_beta or 0.0).to(device)
    return L.build_criterion(cfg.loss, **kw).to(device)


def train_one_epoch(model, loader, criterion, optimizer, scheduler, scaler, cfg, device, ema=None):
    M.set_train_mode(model)
    tot, n = 0.0, 0
    for x, y, _ in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast("cuda", dtype=torch.float16, enabled=cfg.amp):
            if cfg.mix:
                x, tg = L.mix_batch(x, y, cfg.mix_alpha, cfg.mix)
                loss = L.mixed_loss(criterion, model(x), tg)
            else:
                loss = criterion(model(x), y)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        scaler.step(optimizer); scaler.update(); scheduler.step()
        if ema: ema.update(model)
        tot += loss.item() * x.size(0); n += x.size(0)
    return {"train_loss": tot / n, "lr": optimizer.param_groups[-1]["lr"]}


@torch.inference_mode()
def evaluate(model, loader, criterion, device, amp=True):
    model.eval()
    fns, ys, lgs, tot = [], [], [], 0.0
    for x, y, f in loader:
        x, y = x.to(device), y.to(device)
        with torch.autocast("cuda", dtype=torch.float16, enabled=amp):
            lg = model(x)
        lg = lg.float()
        tot += nn.functional.cross_entropy(lg, y, reduction="sum").item()
        fns += list(f); ys.append(y.cpu().numpy()); lgs.append(lg.cpu().numpy())
    ys, lgs = np.concatenate(ys), np.concatenate(lgs)
    return fns, ys, lgs, tot / len(ys)


def softmax(z):
    z = z - z.max(1, keepdims=True); e = np.exp(z); return e / e.sum(1, keepdims=True)


def plot_curves(history, path, title):
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    h = pd.DataFrame(history); ep = h["epoch"]
    fig, ax = plt.subplots(1, 3, figsize=(15, 4))
    ax[0].plot(ep, h.train_loss, label="train"); ax[0].plot(ep, h.val_loss, label="val")
    ax[0].set(title="Loss", xlabel="epoch", ylabel="loss"); ax[0].legend()
    ax[1].plot(ep, h.val_macro_f1, label="val macro-F1"); ax[1].plot(ep, h.val_top1, label="val top-1")
    ax[1].set(title="Metric val", xlabel="epoch", ylabel="score"); ax[1].legend()
    ax[2].plot(ep, h.lr); ax[2].set(title="LR (cuối epoch)", xlabel="epoch", ylabel="lr")
    fig.suptitle(title); fig.tight_layout()
    Path(path).parent.mkdir(parents=True, exist_ok=True); fig.savefig(path, dpi=120); plt.close(fig)


def run(cfg):
    set_seed(cfg.seed)
    rd = run_dir(cfg); rd.mkdir(parents=True, exist_ok=True)
    json.dump(dataclasses.asdict(cfg), open(rd / "config.json", "w"), indent=1)
    dev = torch.device("cuda")
    tr, va, te = D.load_split(cfg.labels_dir, cfg.fold)
    D.check_split(tr, va, te, cfg.images_dir)
    tt, vt = D.build_transforms(True, cfg.img_size, cfg.aug), D.build_transforms(False, cfg.img_size)
    ltr = D.make_loader(tr, cfg.images_dir, tt, cfg.batch_size, True, cfg.sampler, cfg.num_workers, cfg.seed, cfg.cache)
    lva = D.make_loader(va, cfg.images_dir, vt, 128, False, None, cfg.num_workers, cfg.seed, cfg.cache)
    model = M.build_model(cfg.backbone, True, 9, cfg.drop_rate, cfg.init).to(dev).to(memory_format=torch.channels_last)
    crit = build_crit(cfg, tr, dev)
    opt = build_optimizer(model, cfg); sch = build_scheduler(opt, cfg, len(ltr))
    scaler = torch.cuda.amp.GradScaler(enabled=cfg.amp)
    ema = EMA(model, cfg.ema_decay) if cfg.ema_decay else None
    ev_model = lambda: ema.module if ema else model

    hist, best, best_ep, t_ep = [], -1.0, -1, []
    for ep in range(1, cfg.epochs + 1):
        t0 = time.time()
        tl = train_one_epoch(model, ltr, crit, opt, sch, scaler, cfg, dev, ema)
        torch.cuda.synchronize(); t_ep.append(time.time() - t0)
        fns, y, lg, vl = evaluate(ev_model(), lva, crit if False else nn.CrossEntropyLoss(), dev, cfg.amp)
        p = softmax(lg); m = compute_metrics(y, p.argmax(1), p)
        hist.append({"epoch": ep, **tl, "val_loss": vl, "val_macro_f1": m["macro_f1"], "val_top1": m["top1"], "time": t_ep[-1]})
        print(f"[{cfg.exp_id} s{cfg.seed}] ep{ep} train_loss {tl['train_loss']:.4f} val_loss {vl:.4f} F1 {m['macro_f1']:.4f} acc {m['top1']:.4f} ({t_ep[-1]:.0f}s)")
        if m["macro_f1"] > best:  # hòa -> giữ epoch sớm hơn
            best, best_ep = m["macro_f1"], ep
            torch.save(ev_model().state_dict(), rd / "best.pt")
        pd.DataFrame(hist).to_csv(rd / "history.csv", index=False)

    final = copy.deepcopy(ev_model()); final.load_state_dict(torch.load(rd / "best.pt"))
    fns, y, lg, _ = evaluate(final, lva, nn.CrossEntropyLoss(), dev, cfg.amp)
    np.save(rd / "val_logits.npy", lg)
    save_predictions(pred_path(cfg, "val"), fns, y, softmax(lg))
    if cfg.save_test_predictions:  # CHỈ Bước 4
        lte = D.make_loader(te, cfg.images_dir, vt, 128, False, None, cfg.num_workers, cfg.seed, cfg.cache)
        fns, yt, lgt, _ = evaluate(final, lte, nn.CrossEntropyLoss(), dev, cfg.amp)
        np.save(rd / "test_logits.npy", lgt)
        save_predictions(pred_path(cfg, "test"), fns, yt, softmax(lgt))
    plot_curves(hist, Path(cfg.curves_dir) / f"{cfg.exp_id}_seed{cfg.seed}_{cfg.backbone}.png",
                f"{cfg.exp_id} {cfg.backbone} seed{cfg.seed}")
    res = {"exp_id": cfg.exp_id, "backbone": cfg.backbone, "seed": cfg.seed, "best_epoch": best_ep,
           "val_macro_f1": best, "sec_per_epoch": float(np.mean(t_ep)),
           "params_M": M.count_params(model), "weights_tag": getattr(model, "weights_tag", "")}
    json.dump(res, open(rd / "result.json", "w"), indent=1)
    return res

Overwriting /content/drive/MyDrive/deepweeds/code/train.py


In [33]:
import sys, importlib
sys.path.insert(0, "/content/drive/MyDrive/deepweeds/code")
import train as TR; importlib.reload(TR)
cfg = TR.Config(exp_id="DEBUG", backbone="resnet50", epochs=2)
res = TR.run(cfg); print(res)

n: {'train': 10501, 'val': 3501, 'test': 3507} | tổng: 17509 | tỉ lệ: {'train': 0.5997, 'val': 0.2, 'test': 0.2003}
train {0: 675, 1: 637, 2: 618, 3: 613, 4: 637, 5: 605, 6: 644, 7: 609, 8: 5463}
val {0: 225, 1: 213, 2: 206, 3: 204, 4: 212, 5: 202, 6: 215, 7: 203, 8: 1821}
test {0: 226, 1: 213, 2: 207, 3: 205, 4: 213, 5: 202, 6: 215, 7: 204, 8: 1822}
giao: {'train&val': 0, 'train&test': 0, 'val&test': 0} | hợp: 17509 | thiếu file: 0


/content/drive/MyDrive/deepweeds/code/train.py:159: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=cfg.amp)


[DEBUG s0] ep1 train_loss 1.6401 val_loss 1.2596 F1 0.2161 acc 0.5630 (39s)
[DEBUG s0] ep2 train_loss 0.9709 val_loss 0.9164 F1 0.5198 acc 0.6872 (40s)
{'exp_id': 'DEBUG', 'backbone': 'resnet50', 'seed': 0, 'best_epoch': 2, 'val_macro_f1': 0.5197738725651819, 'sec_per_epoch': 39.107630372047424, 'params_M': 23.526473, 'weights_tag': 'timm/resnet50.a1_in1k'}


In [34]:
import torch, dataset as D, model as M, losses as L
tr, va, te = D.load_split("/content/data", 0)
ld = D.make_loader(tr, "/content/data", D.build_transforms(False), 16, False, num_workers=0)
x, y, _ = next(iter(ld)); x, y = x.cuda(), y.cuda()
m = M.build_model("resnet50", True, 9).cuda().train()
opt = torch.optim.AdamW(m.parameters(), lr=3e-4)
for i in range(60):
    opt.zero_grad(); loss = torch.nn.functional.cross_entropy(m(x), y); loss.backward(); opt.step()
    if i % 10 == 0 or i == 59: print(i, round(loss.item(), 4))

0 2.1993
10 0.9438
20 0.3331
30 0.0977
40 0.0312
50 0.0145
59 0.0093


In [35]:
import sys, importlib, json, torch
sys.path.insert(0, "/content/drive/MyDrive/deepweeds/code")
import train as TR; importlib.reload(TR)

BB = [("B01","resnet50"), ("B02","resnext50_32x4d"), ("B03","convnext_tiny"),
      ("B04","swin_tiny_patch4_window7_224"), ("B05","efficientnet_b0")]

for eid, name in BB:
    cfg = TR.Config(exp_id=eid, backbone=name, seed=0, epochs=10)
    done = TR.run_dir(cfg) / "result.json"
    if done.exists():
        print(eid, "đã xong, bỏ qua"); continue
    try:
        res = TR.run(cfg); print(res)
    except Exception as e:
        print(eid, "LỖI:", repr(e))
    torch.cuda.empty_cache()

n: {'train': 10501, 'val': 3501, 'test': 3507} | tổng: 17509 | tỉ lệ: {'train': 0.5997, 'val': 0.2, 'test': 0.2003}
train {0: 675, 1: 637, 2: 618, 3: 613, 4: 637, 5: 605, 6: 644, 7: 609, 8: 5463}
val {0: 225, 1: 213, 2: 206, 3: 204, 4: 212, 5: 202, 6: 215, 7: 203, 8: 1821}
test {0: 226, 1: 213, 2: 207, 3: 205, 4: 213, 5: 202, 6: 215, 7: 204, 8: 1822}
giao: {'train&val': 0, 'train&test': 0, 'val&test': 0} | hợp: 17509 | thiếu file: 0


/content/drive/MyDrive/deepweeds/code/train.py:159: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=cfg.amp)


[B01 s0] ep1 train_loss 1.6401 val_loss 1.2596 F1 0.2161 acc 0.5630 (42s)
[B01 s0] ep2 train_loss 0.9243 val_loss 0.7737 F1 0.5857 acc 0.7252 (38s)
[B01 s0] ep3 train_loss 0.6371 val_loss 0.6301 F1 0.6820 acc 0.7801 (40s)
[B01 s0] ep4 train_loss 0.5036 val_loss 0.5573 F1 0.7284 acc 0.8092 (39s)
[B01 s0] ep5 train_loss 0.4350 val_loss 0.5124 F1 0.7580 acc 0.8283 (39s)
[B01 s0] ep6 train_loss 0.3819 val_loss 0.4863 F1 0.7750 acc 0.8389 (39s)
[B01 s0] ep7 train_loss 0.3488 val_loss 0.4738 F1 0.7804 acc 0.8443 (39s)
[B01 s0] ep8 train_loss 0.3330 val_loss 0.4615 F1 0.7841 acc 0.8443 (40s)
[B01 s0] ep9 train_loss 0.3271 val_loss 0.4694 F1 0.7813 acc 0.8432 (40s)
[B01 s0] ep10 train_loss 0.3197 val_loss 0.4597 F1 0.7846 acc 0.8440 (38s)
{'exp_id': 'B01', 'backbone': 'resnet50', 'seed': 0, 'best_epoch': 10, 'val_macro_f1': 0.7845853620639187, 'sec_per_epoch': 39.51793892383576, 'params_M': 23.526473, 'weights_tag': 'timm/resnet50.a1_in1k'}
n: {'train': 10501, 'val': 3501, 'test': 3507} | tổng

model.safetensors: reconstructing file:   0%|          |  0.00B /  100MB            

model.safetensors: downloading bytes:           |  0.00B            

...

[B04 s0] ep1 train_loss 0.9278 val_loss 0.5237 F1 0.7588 acc 0.8249 (67s)
[B04 s0] ep2 train_loss 0.3122 val_loss 0.3342 F1 0.8471 acc 0.8932 (65s)
[B04 s0] ep3 train_loss 0.2284 val_loss 0.3160 F1 0.8639 acc 0.9000 (65s)
[B04 s0] ep4 train_loss 0.1656 val_loss 0.2243 F1 0.9096 acc 0.9337 (64s)
[B04 s0] ep5 train_loss 0.1198 val_loss 0.1699 F1 0.9359 acc 0.9532 (64s)
[B04 s0] ep6 train_loss 0.0890 val_loss 0.2076 F1 0.9192 acc 0.9383 (64s)
[B04 s0] ep7 train_loss 0.0544 val_loss 0.1333 F1 0.9488 acc 0.9626 (63s)
[B04 s0] ep8 train_loss 0.0420 val_loss 0.1399 F1 0.9425 acc 0.9589 (63s)
[B04 s0] ep9 train_loss 0.0295 val_loss 0.1197 F1 0.9528 acc 0.9652 (63s)
[B04 s0] ep10 train_loss 0.0249 val_loss 0.1236 F1 0.9502 acc 0.9632 (63s)
{'exp_id': 'B04', 'backbone': 'swin_tiny_patch4_window7_224', 'seed': 0, 'best_epoch': 9, 'val_macro_f1': 0.9528089165114425, 'sec_per_epoch': 64.06981840133668, 'params_M': 27.526275, 'weights_tag': 'timm/swin_tiny_patch4_window7_224.ms_in1k'}
n: {'train': 1

model.safetensors: reconstructing file:   0%|          |  0.00B / 21.4MB            

model.safetensors: downloading bytes:           |  0.00B            

/content/drive/MyDrive/deepweeds/code/train.py:159: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=cfg.amp)


[B05 s0] ep1 train_loss 2.0172 val_loss 1.2603 F1 0.4676 acc 0.6075 (39s)
[B05 s0] ep2 train_loss 0.6414 val_loss 0.9094 F1 0.6096 acc 0.7078 (34s)
[B05 s0] ep3 train_loss 0.3774 val_loss 0.7484 F1 0.6689 acc 0.7552 (33s)
[B05 s0] ep4 train_loss 0.2763 val_loss 0.6123 F1 0.7366 acc 0.8069 (33s)
[B05 s0] ep5 train_loss 0.2050 val_loss 0.5906 F1 0.7544 acc 0.8155 (33s)
[B05 s0] ep6 train_loss 0.1576 val_loss 0.5426 F1 0.7713 acc 0.8315 (33s)
[B05 s0] ep7 train_loss 0.1282 val_loss 0.4805 F1 0.8008 acc 0.8512 (33s)
[B05 s0] ep8 train_loss 0.1097 val_loss 0.5061 F1 0.7906 acc 0.8435 (34s)
[B05 s0] ep9 train_loss 0.1014 val_loss 0.5076 F1 0.7883 acc 0.8452 (33s)
[B05 s0] ep10 train_loss 0.0943 val_loss 0.4932 F1 0.7941 acc 0.8480 (35s)
{'exp_id': 'B05', 'backbone': 'efficientnet_b0', 'seed': 0, 'best_epoch': 7, 'val_macro_f1': 0.8007662184773384, 'sec_per_epoch': 34.11805663108826, 'params_M': 4.019077, 'weights_tag': 'timm/efficientnet_b0.ra_in1k'}


In [36]:
import sys; sys.path.insert(0, "/content/drive/MyDrive/deepweeds/code")
import torch, time, model as M
rows = []
for eid, name in [("B01","resnet50"),("B02","resnext50_32x4d"),("B03","convnext_tiny"),
                  ("B04","swin_tiny_patch4_window7_224"),("B05","efficientnet_b0")]:
    m = M.build_model(name, False, 9).cuda().eval()
    g = M.count_gmacs(m, 224)
    x = torch.randn(1,3,224,224).cuda()
    with torch.inference_mode():
        for _ in range(10): m(x)
        torch.cuda.synchronize(); t=[]
        for _ in range(50):
            s=time.perf_counter(); m(x); torch.cuda.synchronize(); t.append((time.perf_counter()-s)*1000)
    t.sort(); rows.append((eid, name, round(g,2), round(t[25],1), round(t[47],1)))
    print(rows[-1]); del m; torch.cuda.empty_cache()

('B01', 'resnet50', 4.11, 24.3, 29.3)
('B02', 'resnext50_32x4d', 4.26, 8.5, 12.3)
('B03', 'convnext_tiny', 4.47, 6.2, 6.6)
('B04', 'swin_tiny_patch4_window7_224', 4.51, 13.4, 17.7)
('B05', 'efficientnet_b0', 0.4, 10.8, 14.7)


In [37]:
import sys; sys.path.insert(0, "/content/drive/MyDrive/deepweeds/code")
import torch, time, numpy as np, model as M
torch.backends.cudnn.benchmark = False
def lat(name, n=200, w=30):
    m = M.build_model(name, False, 9).cuda().eval()
    x = torch.randn(1,3,224,224).cuda(); t=[]
    with torch.inference_mode():
        for _ in range(w): m(x)
        torch.cuda.synchronize()
        for _ in range(n):
            s=time.perf_counter(); m(x); torch.cuda.synchronize(); t.append((time.perf_counter()-s)*1000)
    del m; torch.cuda.empty_cache()
    return [round(float(np.percentile(t,q)),1) for q in (50,95,99)]
for eid, name in [("B01","resnet50"),("B01 lần 2","resnet50"),("B02","resnext50_32x4d"),
                  ("B03","convnext_tiny"),("B04","swin_tiny_patch4_window7_224"),("B05","efficientnet_b0")]:
    print(eid, name, "p50/p95/p99 =", lat(name))

B01 resnet50 p50/p95/p99 = [6.4, 11.5, 12.7]
B01 lần 2 resnet50 p50/p95/p99 = [6.5, 7.0, 10.2]
B02 resnext50_32x4d p50/p95/p99 = [8.3, 10.6, 13.7]
B03 convnext_tiny p50/p95/p99 = [6.2, 7.4, 10.5]
B04 swin_tiny_patch4_window7_224 p50/p95/p99 = [10.0, 11.6, 16.0]
B05 efficientnet_b0 p50/p95/p99 = [10.7, 14.4, 16.7]


In [38]:
import sys, importlib, torch
sys.path.insert(0, "/content/drive/MyDrive/deepweeds/code")
import train as TR; importlib.reload(TR)

BASE = dict(backbone="convnext_tiny", seed=0, epochs=10)
EXPS = [
    ("T01", dict(init="scratch")),
    ("T02", dict(init="frozen")),
    ("T03", dict(aug="color")),
    ("T04", dict(mix="cutmix")),
    ("T05", dict(loss="ls", label_smoothing=0.1)),
    ("T06", dict(loss="ce_weighted", class_weight_beta=0.0)),
]
for eid, ch in EXPS:
    cfg = TR.Config(exp_id=eid, **{**BASE, **ch})
    if (TR.run_dir(cfg) / "result.json").exists():
        print(eid, "đã xong, bỏ qua"); continue
    try:
        print(eid, ch, TR.run(cfg))
    except Exception as e:
        print(eid, "LỖI:", repr(e))
    torch.cuda.empty_cache()

n: {'train': 10501, 'val': 3501, 'test': 3507} | tổng: 17509 | tỉ lệ: {'train': 0.5997, 'val': 0.2, 'test': 0.2003}
train {0: 675, 1: 637, 2: 618, 3: 613, 4: 637, 5: 605, 6: 644, 7: 609, 8: 5463}
val {0: 225, 1: 213, 2: 206, 3: 204, 4: 212, 5: 202, 6: 215, 7: 203, 8: 1821}
test {0: 226, 1: 213, 2: 207, 3: 205, 4: 213, 5: 202, 6: 215, 7: 204, 8: 1822}
giao: {'train&val': 0, 'train&test': 0, 'val&test': 0} | hợp: 17509 | thiếu file: 0


/content/drive/MyDrive/deepweeds/code/train.py:159: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=cfg.amp)
/content/drive/MyDrive/deepweeds/code/train.py:108: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(optimizer); scaler.update(); scheduler.step()


[T01 s0] ep1 train_loss 1.6856 val_loss 1.5658 F1 0.1083 acc 0.5247 (52s)
[T01 s0] ep2 train_loss 1.5895 val_loss 1.5925 F1 0.1308 acc 0.5233 (50s)
[T01 s0] ep3 train_loss 1.5138 val_loss 1.4604 F1 0.1725 acc 0.5330 (50s)
[T01 s0] ep4 train_loss 1.4629 val_loss 1.4123 F1 0.2036 acc 0.5461 (51s)
[T01 s0] ep5 train_loss 1.3689 val_loss 1.4489 F1 0.2240 acc 0.5090 (52s)
[T01 s0] ep6 train_loss 1.3359 val_loss 1.4810 F1 0.2547 acc 0.5139 (51s)
[T01 s0] ep7 train_loss 1.3083 val_loss 1.3648 F1 0.2639 acc 0.5413 (50s)
[T01 s0] ep8 train_loss 1.2690 val_loss 1.3848 F1 0.2821 acc 0.5304 (51s)
[T01 s0] ep9 train_loss 1.2510 val_loss 1.4563 F1 0.2656 acc 0.5173 (50s)
[T01 s0] ep10 train_loss 1.2423 val_loss 1.4188 F1 0.2759 acc 0.5296 (51s)
T01 {'init': 'scratch'} {'exp_id': 'T01', 'backbone': 'convnext_tiny', 'seed': 0, 'best_epoch': 8, 'val_macro_f1': 0.28211741462498435, 'sec_per_epoch': 50.81786432266235, 'params_M': 27.827049, 'weights_tag': 'scratch'}
n: {'train': 10501, 'val': 3501, 'test

/content/drive/MyDrive/deepweeds/code/train.py:159: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=cfg.amp)
/content/drive/MyDrive/deepweeds/code/train.py:108: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(optimizer); scaler.update(); scheduler.step()


[T02 s0] ep1 train_loss 1.1452 val_loss 0.6338 F1 0.7441 acc 0.7989 (29s)
[T02 s0] ep2 train_loss 0.5212 val_loss 0.4701 F1 0.8113 acc 0.8515 (30s)
[T02 s0] ep3 train_loss 0.4201 val_loss 0.4465 F1 0.8102 acc 0.8520 (30s)
[T02 s0] ep4 train_loss 0.3721 val_loss 0.4205 F1 0.8211 acc 0.8603 (29s)
[T02 s0] ep5 train_loss 0.3405 val_loss 0.4100 F1 0.8391 acc 0.8703 (29s)
[T02 s0] ep6 train_loss 0.3271 val_loss 0.3865 F1 0.8422 acc 0.8755 (29s)
[T02 s0] ep7 train_loss 0.3122 val_loss 0.3740 F1 0.8446 acc 0.8769 (29s)
[T02 s0] ep8 train_loss 0.3047 val_loss 0.3721 F1 0.8455 acc 0.8763 (29s)
[T02 s0] ep9 train_loss 0.2958 val_loss 0.3712 F1 0.8459 acc 0.8780 (29s)
[T02 s0] ep10 train_loss 0.2944 val_loss 0.3699 F1 0.8455 acc 0.8775 (29s)
T02 {'init': 'frozen'} {'exp_id': 'T02', 'backbone': 'convnext_tiny', 'seed': 0, 'best_epoch': 9, 'val_macro_f1': 0.8459430406243835, 'sec_per_epoch': 29.307338142395018, 'params_M': 27.827049, 'weights_tag': 'timm/convnext_tiny.in12k_ft_in1k'}
n: {'train': 1

...

[T04 s0] ep1 train_loss 1.1235 val_loss 0.5229 F1 0.7573 acc 0.8303 (52s)
[T04 s0] ep2 train_loss 0.7976 val_loss 0.2183 F1 0.9140 acc 0.9340 (50s)
[T04 s0] ep3 train_loss 0.6708 val_loss 0.2128 F1 0.9130 acc 0.9343 (51s)
[T04 s0] ep4 train_loss 0.6446 val_loss 0.1548 F1 0.9511 acc 0.9580 (51s)
[T04 s0] ep5 train_loss 0.5889 val_loss 0.1445 F1 0.9449 acc 0.9583 (51s)
[T04 s0] ep6 train_loss 0.5474 val_loss 0.0954 F1 0.9609 acc 0.9712 (50s)
[T04 s0] ep7 train_loss 0.5007 val_loss 0.1011 F1 0.9640 acc 0.9726 (51s)
[T04 s0] ep8 train_loss 0.4805 val_loss 0.0869 F1 0.9665 acc 0.9746 (51s)
[T04 s0] ep9 train_loss 0.4671 val_loss 0.0773 F1 0.9699 acc 0.9769 (50s)
[T04 s0] ep10 train_loss 0.4452 val_loss 0.0759 F1 0.9693 acc 0.9769 (51s)
T04 {'mix': 'cutmix'} {'exp_id': 'T04', 'backbone': 'convnext_tiny', 'seed': 0, 'best_epoch': 9, 'val_macro_f1': 0.9698565428502496, 'sec_per_epoch': 50.86682643890381, 'params_M': 27.827049, 'weights_tag': 'timm/convnext_tiny.in12k_ft_in1k'}
n: {'train': 105

/content/drive/MyDrive/deepweeds/code/train.py:159: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=cfg.amp)
/content/drive/MyDrive/deepweeds/code/train.py:108: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(optimizer); scaler.update(); scheduler.step()


[T05 s0] ep1 train_loss 1.0070 val_loss 0.5186 F1 0.7764 acc 0.8426 (53s)
[T05 s0] ep2 train_loss 0.6863 val_loss 0.3142 F1 0.8885 acc 0.9206 (49s)
[T05 s0] ep3 train_loss 0.6385 val_loss 0.3163 F1 0.9174 acc 0.9312 (51s)
[T05 s0] ep4 train_loss 0.6004 val_loss 0.3128 F1 0.8970 acc 0.9169 (50s)
[T05 s0] ep5 train_loss 0.5609 val_loss 0.2324 F1 0.9387 acc 0.9532 (50s)
[T05 s0] ep6 train_loss 0.5389 val_loss 0.1836 F1 0.9566 acc 0.9666 (50s)
[T05 s0] ep7 train_loss 0.5156 val_loss 0.1830 F1 0.9563 acc 0.9674 (50s)
[T05 s0] ep8 train_loss 0.5011 val_loss 0.1900 F1 0.9554 acc 0.9669 (50s)
[T05 s0] ep9 train_loss 0.4949 val_loss 0.1780 F1 0.9608 acc 0.9694 (50s)
[T05 s0] ep10 train_loss 0.4932 val_loss 0.1759 F1 0.9629 acc 0.9712 (50s)
T05 {'loss': 'ls', 'label_smoothing': 0.1} {'exp_id': 'T05', 'backbone': 'convnext_tiny', 'seed': 0, 'best_epoch': 10, 'val_macro_f1': 0.9628700807460606, 'sec_per_epoch': 50.24998424053192, 'params_M': 27.827049, 'weights_tag': 'timm/convnext_tiny.in12k_ft_i

/content/drive/MyDrive/deepweeds/code/train.py:159: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=cfg.amp)
/content/drive/MyDrive/deepweeds/code/train.py:108: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(optimizer); scaler.update(); scheduler.step()


[T06 s0] ep1 train_loss 0.8086 val_loss 0.5775 F1 0.8015 acc 0.8143 (52s)
[T06 s0] ep2 train_loss 0.3616 val_loss 0.6299 F1 0.8108 acc 0.8089 (50s)
[T06 s0] ep3 train_loss 0.2505 val_loss 0.6326 F1 0.7952 acc 0.7923 (50s)
[T06 s0] ep4 train_loss 0.1697 val_loss 0.2351 F1 0.9122 acc 0.9266 (50s)
[T06 s0] ep5 train_loss 0.1244 val_loss 0.3176 F1 0.8952 acc 0.9046 (50s)
[T06 s0] ep6 train_loss 0.0730 val_loss 0.2048 F1 0.9319 acc 0.9443 (49s)
[T06 s0] ep7 train_loss 0.0352 val_loss 0.1800 F1 0.9452 acc 0.9537 (49s)
[T06 s0] ep8 train_loss 0.0222 val_loss 0.1520 F1 0.9539 acc 0.9634 (49s)
[T06 s0] ep9 train_loss 0.0121 val_loss 0.1312 F1 0.9610 acc 0.9689 (49s)
[T06 s0] ep10 train_loss 0.0076 val_loss 0.1313 F1 0.9591 acc 0.9677 (48s)
T06 {'loss': 'ce_weighted', 'class_weight_beta': 0.0} {'exp_id': 'T06', 'backbone': 'convnext_tiny', 'seed': 0, 'best_epoch': 9, 'val_macro_f1': 0.9610291093078865, 'sec_per_epoch': 49.6555061340332, 'params_M': 27.827049, 'weights_tag': 'timm/convnext_tiny.i

In [3]:
!rm -rf /content/drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
!ls /content/drive/MyDrive/deepweeds
!ls /content/drive/MyDrive/deepweeds/runs

code  curves  predictions  runs
B01  B02  B03  B04  B05  DEBUG	T01  T02  T03  T04  T05  T06


In [7]:
!mkdir -p /content/drive/MyDrive/deepweeds/runs/T00
!cp -r /content/drive/MyDrive/deepweeds/runs/B03/seed0 /content/drive/MyDrive/deepweeds/runs/T00/
!ls /content/drive/MyDrive/deepweeds/runs/T00/seed0

best.pt  config.json  history.csv  result.json	val_logits.npy


In [9]:
!pip install -q timm fvcore
import torch; print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 1.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 6.0 MB/s eta 0:00:00
False 
